# Notebook 01 — Exploratory Data Analysis

Return distributions, correlation heatmaps, volatility regimes, and missing-data audit for the US ETF universe and India NSE large-cap universe.

**QM640 Data Analytics Capstone · Walsh College · Anupam K Mitra · May 2026**

---
> **Standalone notebook** — all code is self-contained. Run cells top-to-bottom. Data is downloaded automatically on first run.

## 1. Imports & Configuration

In [ ]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# ── Universe ──────────────────────────────────────────────────
US_TICKERS   = ['SPY','QQQ','IWM','GLD','TLT','HYG','IEF',
                 'USO','XLF','XLE','XLK','BTC-USD']
IND_TICKERS  = ['^NSEI','^NSEBANK','TCS.NS','INFY.NS','HDFCBANK.NS',
                 'SBIN.NS','SUNPHARMA.NS','HINDUNILVR.NS','ONGC.NS',
                 'COALINDIA.NS','USDINR=X','GC=F','CL=F','SPY','^INDIAVIX']
START, END_US, END_IN = '2010-01-01','2024-12-31','2026-04-30'
os.makedirs('../results/figures', exist_ok=True)
print('Setup complete')

## 2. Download Data

In [ ]:
import yfinance as yf, time

def download_group(tickers, start, end, label):
    print(f'  Downloading {label} ...')
    try:
        raw = yf.download(tickers, start=start, end=end,
                          auto_adjust=True, progress=False,
                          threads=False, timeout=30)
        prices = (raw['Close'] if isinstance(raw.columns, pd.MultiIndex)
                  else raw)
    except Exception as e:
        print(f'  batch failed: {e}'); prices = pd.DataFrame()
    results = {}
    for t in tickers:
        col = prices[t] if t in prices.columns else None
        if col is not None and col.notna().mean() > 0.5:
            results[t] = col
        else:
            time.sleep(0.4)
            try:
                ind = yf.download(t, start=start, end=end,
                                   auto_adjust=True, progress=False)
                s = (ind['Close'] if isinstance(ind.columns, pd.MultiIndex)
                     else ind.iloc[:,0])
                if s.notna().mean() > 0.3: results[t] = s.squeeze()
            except: pass
    df = pd.concat(results, axis=1).ffill(limit=3) if results else pd.DataFrame()
    print(f'  {label}: {df.shape[1]} tickers, {len(df)} days')
    return df

prices_us  = download_group(US_TICKERS,  START, END_US,  'US ETF')
prices_ind = download_group(IND_TICKERS, START, END_IN, 'India NSE')
print('Done.')

## 3. Return Statistics

In [ ]:
def log_ret(p):
    return np.log(p / p.shift(1)).clip(-0.20, 0.20)

r_us  = log_ret(prices_us).dropna(how='all')
r_ind = log_ret(prices_ind).dropna(how='all')

print('US — Annualised return and vol:')
stats_us = pd.DataFrame({
    'Ann. Return':  r_us.mean() * 252,
    'Ann. Vol':     r_us.std()  * np.sqrt(252),
    'Sharpe':      (r_us.mean() * 252 - 0.04) / (r_us.std() * np.sqrt(252)),
    'Skewness':     r_us.skew(),
    'Kurtosis':     r_us.kurtosis(),
}).round(4)
print(stats_us.to_string())

## 4. Return Distribution Plots

In [ ]:
key_assets = {
    'SPY (US Benchmark)':    r_us.get('SPY'),
    'GLD (Gold)':            r_us.get('GLD'),
    '^NSEI (Nifty 50)':      r_ind.get('^NSEI'),
    'USDINR=X (USD/INR)':    r_ind.get('USDINR=X'),
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colours = ['#1A9988','#CA8A04','#1C3678','#DC2626']
for ax, (label, r), col in zip(axes.flat, key_assets.items(), colours):
    if r is None: continue
    r_clean = r.dropna()
    ax.hist(r_clean, bins=80, color=col, alpha=0.65, density=True)
    xs = np.linspace(r_clean.min(), r_clean.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, r_clean.mean(), r_clean.std()),
            'k--', lw=1.5, label='Normal fit')
    ax.set_title(f'{label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Log Return'); ax.legend(fontsize=8)
    ax.annotate(f'skew={r_clean.skew():.2f}  kurt={r_clean.kurtosis():.2f}',
                xy=(0.02,0.92), xycoords='axes fraction', fontsize=9)
fig.suptitle('Return Distributions — Fat Tails vs Normal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/01_return_distributions.png', dpi=150)
plt.show(); print('Fig 1 saved')

## 5. Correlation Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, r, title in zip(axes,
        [r_us, r_ind[['NSEI' if 'NSEI' in r_ind else r_ind.columns[0],
                       '^NSEBANK','INFY.NS','HDFCBANK.NS','SBIN.NS',
                       'USDINR=X','GC=F','CL=F','SPY']
                      if all(c in r_ind.columns for c in
                             ['^NSEBANK','INFY.NS','GC=F']) else r_ind]],
        ['US ETF Universe', 'India NSE Universe']):
    corr = r.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7},
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{title} — Pairwise Correlation', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/01_correlation_heatmaps.png', dpi=150)
plt.show(); print('Fig 2 saved')

## 6. Volatility Regime Timeline

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
crisis_bands = {
    'GFC 2008-09': ('2008-09-01','2009-03-31','#E24B4A'),
    'COVID 2020':  ('2020-02-01','2020-06-30','#9B59B6'),
    'Rate shock 2022': ('2022-01-01','2022-12-31','#EF9F27'),
}
india_bands = {
    'Demonet. 2016': ('2016-11-01','2017-01-31','#E24B4A'),
    'NBFC 2018':     ('2018-08-01','2019-01-31','#EF9F27'),
    'COVID 2020':    ('2020-02-01','2020-06-30','#9B59B6'),
    'FII outflow 22':('2022-01-01','2022-10-31','#CA8A04'),
}
for ax, r, label, bands in zip(axes,
        [r_us.get('SPY'), r_ind.get('^NSEI')],
        ['SPY (US)', 'Nifty 50 (India)'],
        [crisis_bands, india_bands]):
    if r is None: continue
    rvol = r.rolling(21).std() * np.sqrt(252) * 100
    ax.fill_between(rvol.index, rvol, alpha=0.35, color='#EB5600')
    ax.plot(rvol.index, rvol, color='#EB5600', lw=0.8)
    for name, (s, e, col) in bands.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.18, color=col)
        ax.text(pd.Timestamp(s), rvol.max()*0.88, name,
                fontsize=7.5, color=col, rotation=90, ha='right')
    ax.set_title(f'{label} — 21-Day Realised Volatility (%)', fontweight='bold')
    ax.set_ylabel('Ann. Vol %')
plt.tight_layout()
plt.savefig('../results/figures/01_volatility_regimes.png', dpi=150)
plt.show(); print('Fig 3 saved')

## 7. Missing Data Audit

In [ ]:
print('US ETF — Missing % per ticker:')
miss_us = (prices_us.isna().mean() * 100).sort_values(ascending=False)
print(miss_us.round(2).to_string())

print('\nIndia NSE — Missing % per ticker:')
miss_in = (prices_ind.isna().mean() * 100).sort_values(ascending=False)
print(miss_in.round(2).to_string())

print(f'\nUS:    {prices_us.shape[0]} trading days, {prices_us.shape[1]} tickers')
print(f'India: {prices_ind.shape[0]} trading days, {prices_ind.shape[1]} tickers')